# Phase 1: Exploratory Data Analysis & Data Alignment

In this notebook, we document our step-by-step exploratory data analysis, discover alignment discrepancies between temporal resolutions, diagnose raw telemetry clock-drift issues, and establish a robust re-aggregation pipeline. We then perform distribution and outlier analysis.

In [ ]:
# CELL 1: Initial Loading & Structural Profiling
import os
import pandas as pd
import numpy as np

base_dir = "/home/zibo127/Documents/institution_subnets"

# Load raw datasets for Subnet 0
df_10m = pd.read_csv(os.path.join(base_dir, "agg_10_minutes", "0.csv"))
df_1h = pd.read_csv(os.path.join(base_dir, "agg_1_hour", "0.csv" ))

print("--- Structural Profiling ---")
print(f"Raw 10m Data Shape: {df_10m.shape}")
print(f"Raw Hourly Data Shape: {df_1h.shape}")
print(f"Raw 10m Columns: {list(df_10m.columns)}")

In [ ]:
# CELL 2: First Naive Consistency Audit
# Reindex to continuous indices and fill gaps via linear interpolation
idx_10m = pd.Index(np.arange(df_10m["id_time"].min(), df_10m["id_time"].max() + 1), name="id_time")
df_10m_clean = df_10m.set_index("id_time").reindex(idx_10m).interpolate(method="linear")

idx_1h = pd.Index(np.arange(df_1h["id_time"].min(), df_1h["id_time"].max() + 1), name="id_time")
df_1h_clean = df_1h.set_index("id_time").reindex(idx_1h).interpolate(method="linear")

# Aggregate 10-minute records to hourly by grouping by index // 6
df_10m_hourly = df_10m_clean.groupby(df_10m_clean.index // 6).agg({
    "n_flows": "sum",
    "n_packets": "sum",
    "n_bytes": "sum"
})

# Align and compute naive correlation
common_hours = df_10m_hourly.index.intersection(df_1h_clean.index)
flows_resampled = df_10m_hourly.loc[common_hours, "n_flows"]
flows_actual = df_1h_clean.loc[common_hours, "n_flows"]

correlation = np.corrcoef(flows_resampled, flows_actual)[0, 1]
mape = np.mean(np.abs(flows_resampled - flows_actual) / (flows_actual + 1)) * 100

print(f"Naive Correlation (r): {correlation:.6f}")
print(f"Naive MAPE: {mape:.6f}%")

all_comparison = pd.DataFrame({
    'Calculated_10m_Sum': flows_resampled,
    'Actual_Hourly': flows_actual
})
all_comparison['Difference'] = all_comparison['Calculated_10m_Sum'] - all_comparison['Actual_Hourly']
mismatches = all_comparison[all_comparison['Difference'] != 0]
print(f"Total mismatched hours found: {len(mismatches)}")

In [ ]:
# CELL 3: Diagnostic - Inspecting the Transition Point
# Print comparison around hour 480 to see where the mismatch begins
print(all_comparison.loc[475:485])

In [ ]:
# CELL 4: Option B Fix Attempt (Shift raw hourly index >= 481 back by 1)
df_1h_shifted = df_1h.copy()
df_1h_shifted.loc[df_1h_shifted['id_time'] >= 481, 'id_time'] -= 1

# Reindex and interpolate shifted hourly dataset
idx_1h_shifted = pd.Index(np.arange(df_1h_shifted["id_time"].min(), df_1h_shifted["id_time"].max() + 1), name="id_time")
df_1h_clean_shifted = df_1h_shifted.set_index("id_time").reindex(idx_1h_shifted).interpolate(method="linear")

# Align and compute comparison again
common_hours_shifted = df_10m_hourly.index.intersection(df_1h_clean_shifted.index)
flows_resampled_s = df_10m_hourly.loc[common_hours_shifted, "n_flows"]
flows_actual_s = df_1h_clean_shifted.loc[common_hours_shifted, "n_flows"]

corr_s = np.corrcoef(flows_resampled_s, flows_actual_s)[0, 1]
mape_s = np.mean(np.abs(flows_resampled_s - flows_actual_s) / (flows_actual_s + 1)) * 100

all_comp_s = pd.DataFrame({
    'Calculated_10m_Sum': flows_resampled_s,
    'Actual_Hourly': flows_actual_s
})
all_comp_s['Difference'] = all_comp_s['Calculated_10m_Sum'] - all_comp_s['Actual_Hourly']
mismatches_s = all_comp_s[all_comp_s['Difference'] != 0]

print(f"Shifted Correlation (r): {corr_s:.6f}")
print(f"Shifted MAPE: {mape_s:.6f}%")
print(f"Remaining mismatched hours: {len(mismatches_s)}")
print("\nFirst 10 remaining mismatches:")
print(mismatches_s.head(10))

In [ ]:
# CELL 5: Diagnostic - Identifying Gaps in Raw Files
# Look for missing indices in raw hourly file
expected_hourly = set(range(df_1h["id_time"].min(), df_1h["id_time"].max() + 1))
actual_hourly = set(df_1h["id_time"])
missing_hourly = sorted(list(expected_hourly - actual_hourly))

# Look for missing indices in raw 10-minute file
expected_10m = set(range(df_10m["id_time"].min(), df_10m["id_time"].max() + 1))
actual_10m = set(df_10m["id_time"])
missing_10m = sorted(list(expected_10m - actual_10m))

print(f"Missing hourly indices in raw file: {missing_hourly}")
print(f"Missing 10-minute indices in raw file: {missing_10m}")

In [ ]:
# CELL 6: Robust Solution - Clean & Aggregate from 10m Ground Truth
def get_clean_subnet_data(subnet_id: int):
    # Load 10m data
    file_path = os.path.join(base_dir, "agg_10_minutes", f"{subnet_id}.csv")
    df = pd.read_csv(file_path)
    
    # Reindex to continuous 10m grid to handle gaps (like 26028, 26029)
    t_min = df["id_time"].min()
    t_max = df["id_time"].max()
    idx = pd.Index(np.arange(t_min, t_max + 1), name="id_time")
    df_clean = df.set_index("id_time").reindex(idx).interpolate(method="linear")
    
    # Define aggregation behavior
    agg_dict = {
        "n_flows": "sum",
        "n_packets": "sum",
        "n_bytes": "sum",
        "avg_duration": "mean",
        "avg_ttl": "mean",
        "tcp_udp_ratio_packets": "mean",
        "tcp_udp_ratio_bytes": "mean",
        "dir_ratio_packets": "mean",
        "dir_ratio_bytes": "mean",
        "average_n_dest_asn": "mean",
        "average_n_dest_ports": "mean",
        "average_n_dest_ip": "mean"
    }
    
    agg_dict = {col: agg for col, agg in agg_dict.items() if col in df_clean.columns}
    df_hourly = df_clean.groupby(df_clean.index // 6).agg(agg_dict)
    df_hourly.index.name = "id_time"
    return df_hourly

# Create clean hourly data for Subnet 0
df_hour_clean = get_clean_subnet_data(0)
print(f"Clean Hourly Data Shape: {df_hour_clean.shape}")

In [ ]:
# CELL 7: Distribution Analysis (Skewness & Kurtosis)
flows_skew_orig = df_hour_clean['n_flows'].skew()
flows_kurt_orig = df_hour_clean['n_flows'].kurt()

bytes_skew_orig = df_hour_clean['n_bytes'].skew()
bytes_kurt_orig = df_hour_clean['n_bytes'].kurt()

# Apply log1p transform
flows_log = np.log1p(df_hour_clean['n_flows'])
bytes_log = np.log1p(df_hour_clean['n_bytes'])

print("--- Distribution Analysis: n_flows ---")
print(f"Original -> Skewness: {flows_skew_orig:.4f}, Excess Kurtosis: {flows_kurt_orig:.4f}")
print(f"Log1p    -> Skewness: {flows_log.skew():.4f}, Excess Kurtosis: {flows_log.kurt():.4f}\n")

print("--- Distribution Analysis: n_bytes ---")
print(f"Original -> Skewness: {bytes_skew_orig:.4f}, Excess Kurtosis: {bytes_kurt_orig:.4f}")
print(f"Log1p    -> Skewness: {bytes_log.skew():.4f}, Excess Kurtosis: {bytes_log.kurt():.4f}")

In [ ]:
# CELL 8: Outlier Analysis (Z-Score vs. IQR)
# Raw data IQR Outliers
Q1_raw = df_hour_clean['n_flows'].quantile(0.25)
Q3_raw = df_hour_clean['n_flows'].quantile(0.75)
IQR_raw = Q3_raw - Q1_raw
upper_raw = Q3_raw + 1.5 * IQR_raw
outliers_iqr_raw = (df_hour_clean['n_flows'] > upper_raw).sum()

# Raw data Z-score Outliers
z_raw = (df_hour_clean['n_flows'] - df_hour_clean['n_flows'].mean()) / df_hour_clean['n_flows'].std()
outliers_z_raw = (z_raw.abs() > 3).sum()

# Log data IQR Outliers
Q1_log = flows_log.quantile(0.25)
Q3_log = flows_log.quantile(0.75)
IQR_log = Q3_log - Q1_log
lower_log = Q1_log - 1.5 * IQR_log
upper_log = Q3_log + 1.5 * IQR_log
outliers_iqr_log = ((flows_log < lower_log) | (flows_log > upper_log)).sum()

# Log data Z-score Outliers
z_log = (flows_log - flows_log.mean()) / flows_log.std()
outliers_z_log = (z_log.abs() > 3).sum()

print("--- Outliers Identified ---")
print(f"IQR (Raw):       {outliers_iqr_raw}")
print(f"Z-Score (Raw):   {outliers_z_raw}")
print(f"IQR (Log1p):     {outliers_iqr_log}")
print(f"Z-Score (Log1p): {outliers_z_log}")

In [ ]:
# CELL 9: Correlation Analysis
corr_matrix = df_hour_clean.corr(method='pearson')
print("--- Correlation Matrix ---")
print(corr_matrix[['n_flows', 'n_packets', 'n_bytes']])

In [ ]:
# CELL 10: 1-Week Traffic Visualization
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))
plt.plot(df_hour_clean.index[:168], df_hour_clean['n_flows'].iloc[:168], label='n_flows', color='indigo', linewidth=1.5)
plt.title('Subnet 0 Traffic: First 168 Hours (1 Week)')
plt.xlabel('Time (Hours)')
plt.ylabel('Flow Count')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Phase 2: Time Series Foundations

In this section, we study statistical stationarity using the Augmented Dickey-Fuller (ADF) test, and explore temporal correlation properties using Autocorrelation (ACF) and Partial Autocorrelation (PACF).

In [ ]:
# CELL 11: Augmented Dickey-Fuller (ADF) Test
from statsmodels.tsa.stattools import adfuller

print("--- Augmented Dickey-Fuller Test on Raw Flows ---")
result = adfuller(df_hour_clean['n_flows'].dropna())

print(f"ADF Statistic: {result[0]:.6f}")
print(f"p-value: {result[1]:.6e}")
print("Critical Values:")
for key, value in result[4].items():
    print(f"\t{key}: {value:.3f}")

In [ ]:
# CELL 12: Autocorrelation (ACF) & Partial Autocorrelation (PACF) Plots (168 Lags)
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Plot Autocorrelation Function (ACF)
plot_acf(df_hour_clean['n_flows'].dropna(), lags=168, ax=axes[0], color='teal', vlines_kwargs={'colors': 'teal'})
axes[0].set_title('Autocorrelation Function (ACF) - 168 Lags')
axes[0].set_xlabel('Lags (Hours)')
axes[0].set_ylabel('Correlation')
axes[0].grid(True, alpha=0.3)

# Plot Partial Autocorrelation Function (PACF)
plot_pacf(df_hour_clean['n_flows'].dropna(), lags=168, ax=axes[1], color='darkorange', vlines_kwargs={'colors': 'darkorange'})
axes[1].set_title('Partial Autocorrelation Function (PACF) - 168 Lags')
axes[1].set_xlabel('Lags (Hours)')
axes[1].set_ylabel('Correlation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()